# E04 — Thermal Expansion & Constrained Thermal Stress
*Exam tool — simplified 3-part structure.*

---

## Part 1 — Theory Recap

### Free (Unconstrained) Expansion
$$\Delta L = L \cdot \alpha \cdot \Delta T$$
The linear CTE $\alpha$ is assumed constant over the temperature range.

### Fully Constrained Thermal Stress
$$\sigma_{\text{thermal}} = E \cdot \alpha \cdot \Delta T$$
Derived from Hooke's law: the prevented strain $\varepsilon_{\text{prevented}} = \alpha \Delta T$ causes stress. The rod is in **compression** when $\Delta T > 0$ (expansion blocked).

### Partial Constraint (Gap Available)
$$F = E \cdot A_{\text{cross}} \cdot \left(\alpha \Delta T - \frac{\delta_{\text{gap}}}{L}\right)$$
If the gap $\delta_{\text{gap}} \geq \Delta L$, the expansion is fully absorbed and $F = 0$.

### Constrained Force (Circular Cross-Section)
$$A_{\text{cross}} = \frac{\pi d^2}{4}$$

### Critical Temperature Rise (Yielding)
$$\Delta T_{\text{crit}} = \frac{\sigma_{\text{yield}}}{E \cdot \alpha}$$
At $\Delta T > \Delta T_{\text{crit}}$ the thermal stress exceeds yield — the part deforms permanently.

### Combined with Press-Fit
When a polymer hub on a steel shaft heats up, the hub expands more than the shaft ($\alpha_{\text{polymer}} > \alpha_{\text{steel}}$), **relieving** the interference. Use:
$$\delta_{\text{effective}} = \delta_0 - (\alpha_{\text{polymer}} - \alpha_{\text{steel}}) \cdot D_1 \cdot \Delta T$$

### Parameter Table
| Symbol | Description | Unit |
|--------|-------------|------|
| $L$ | Rod/component length | m |
| $d$ | Diameter (circular cross-section) | m |
| $\alpha$ | Linear CTE | 1/°C |
| $\Delta T$ | Temperature change ($>0$ = heating) | °C |
| $E$ | Tensile modulus | Pa |
| $\sigma_{\text{yield}}$ | Yield stress | Pa |
| $\delta_{\text{gap}}$ | Available clearance gap | m |

---

## ⚠ Common Exam Pitfalls
1. **Wrong modulus** — use the **tensile** modulus, not the flexural modulus, for thermal stress.
2. **Cross-section area** — compute $A = \pi d^2/4$ before calculating force; don't confuse with area moment.
3. **Sign convention** — constrained expansion creates **compressive** stress in the rod. Report magnitude for yield comparison.
4. **Gap direction** — if the gap absorbs all expansion ($\delta_{\text{gap}} \geq \Delta L$), the force is exactly zero.

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 2 — Problem Inputs  (edit values here)
# ═══════════════════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '..')
import numpy as np

from utils.unit_registry import ureg, Q_, strip_units
from utils.material_db import THERMAL_EXPANSION

# ── Material ─────────────────────────────────────────────────────────────────
MAT_KEY = 'PVC'   # keys: 'PP', 'POM', 'PC', 'PA66', 'PVC'
mat_cte = THERMAL_EXPANSION.get(MAT_KEY, {})
CTE     = mat_cte.get('CTE', 70e-6)   # fallback 70e-6 1/°C if key missing
CTE_src = mat_cte.get('description', 'uPVC (Rigid PVC)')

# ── Geometry ─────────────────────────────────────────────────────────────────
L_rod      = Q_(12000.0, 'mm')    # rod length
d_rod      = Q_(  8.0, 'mm')      # rod diameter (ignored if A_rod is set)
A_rod      = Q_(400.0, 'mm**2')   # direct area input (overrides d_rod if set)
delta_gap  = Q_(  0.0, 'mm')      # available expansion gap (0 = fully constrained)

# ── Thermal loading ───────────────────────────────────────────────────────────
delta_T = 40.0   # [°C] temperature change (positive = heating)

# ── Material properties ───────────────────────────────────────────────────────
E_tensile   = Q_(1500.0, 'MPa')   # tensile modulus (NOT flexural)
sigma_yield = Q_(  30.0, 'MPa')   # yield stress
epsilon_max = 0.008               # maximal usable strain (e.g. 0.8%)

print(f'CTE source   : {CTE_src}')
print(f'CTE [1/°C]   : {CTE:.2e}')


CTE source   : Rigid PVC (uPVC), 23°C
CTE [1/°C]   : 7.00e-05


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Functions
# ═══════════════════════════════════════════════════════════════════════════════

def free_expansion(L_m, alpha, dT):
    """Unconstrained thermal expansion ΔL = L·α·ΔT. Returns metres."""
    return L_m * alpha * dT


def constrained_stress(E_Pa, alpha, dT):
    """Fully constrained thermal stress σ = E·α·ΔT. Returns Pa (magnitude)."""
    return E_Pa * alpha * abs(dT)


def constraint_force(E_Pa, A_m2, alpha, dT, delta_gap_m, L_m):
    """Thermal constraint force with optional gap.
    F = E·A·(α·ΔT - δ_gap/L). Clamped to zero if gap absorbs all expansion."""
    net_strain = alpha * dT - delta_gap_m / L_m
    if net_strain <= 0:
        return 0.0
    return E_Pa * A_m2 * net_strain


print('Functions defined.')


Functions defined.


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Execution
# ═══════════════════════════════════════════════════════════════════════════════

L_m       = strip_units(L_rod.to('m'))
gap_m     = strip_units(delta_gap.to('m'))
E_Pa      = strip_units(E_tensile.to('Pa'))
sy_Pa     = strip_units(sigma_yield.to('Pa'))

if A_rod.magnitude is not None:
    A_cross = strip_units(A_rod.to('m**2'))
    d_m     = np.sqrt(4 * A_cross / np.pi)
else:
    d_m     = strip_units(d_rod.to('m'))
    A_cross = np.pi * d_m**2 / 4.0

direction = 'heating' if delta_T > 0 else 'cooling'

# --- Step 1: Material and geometry -------------------------------------------
print('--- Step 1: Material & Geometry ---')
print(f"  {'Material':<30}: {MAT_KEY}")
print(f"  {'CTE α [μm/(m·°C)]':<30}: {CTE*1e6:.1f}")
print(f"  {'L [mm]':<30}: {L_m*1e3:.1f}")
print(f"  {'A_cross [mm²]':<30}: {A_cross*1e6:.1f}")
print(f"  {'ΔT [°C]':<30}: {delta_T:+.1f}  ({direction})")
print(f"  {'Gap δ_gap [mm]':<30}: {gap_m*1e3:.3f}")

# --- Step 2: Free expansion --------------------------------------------------
dL_free = free_expansion(L_m, CTE, delta_T)
epsilon = dL_free / L_m

print()
print('--- Step 2: Free Expansion & Strain ---')
print(f"  {'ΔL_free = L·α·ΔT [mm]':<30}: {dL_free*1e3:.4f}")
print(f"  {'Strain ε [%]':<30}: {epsilon*100:.4f} %")

# --- Step 3: Constrained stress and force ------------------------------------
sigma_th  = constrained_stress(E_Pa, CTE, delta_T)
F_constr  = constraint_force(E_Pa, A_cross, CTE, delta_T, gap_m, L_m)
SF_yield  = sy_Pa / sigma_th if sigma_th > 0 else float('inf')

print()
print('--- Step 3: Constrained Stress & Force ---')
print(f"  {'σ_thermal [MPa]':<30}: {sigma_th/1e6:.3f}")
print(f"  {'F_constraint [N]':<30}: {F_constr:.1f}")
print(f"  {'Safety factor (yield)':<30}: {SF_yield:.2f}")


--- Step 1: Material & Geometry ---
  Material                      : PVC
  CTE α [μm/(m·°C)]             : 70.0
  L [mm]                        : 12000.0
  A_cross [mm²]                 : 400.0
  ΔT [°C]                       : +40.0  (heating)
  Gap δ_gap [mm]                : 0.000

--- Step 2: Free Expansion & Strain ---
  ΔL_free = L·α·ΔT [mm]         : 33.6000
  Strain ε [%]                  : 0.2800 %

--- Step 3: Constrained Stress & Force ---
  σ_thermal [MPa]               : 4.200
  F_constraint [N]              : 1680.0
  Safety factor (yield)         : 7.14


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 4 — Validation
# ═══════════════════════════════════════════════════════════════════════════════

pass_stress = sigma_th <= sy_Pa
pass_strain = abs(epsilon) <= epsilon_max
overall     = pass_stress and pass_strain

print('--- VALIDATION ---')
print(f"  Thermal stress  : {sigma_th/1e6:.3f} MPa  <=  {sy_Pa/1e6:.1f} MPa (yield)  | {'PASS' if pass_stress else 'FAIL'}")
print(f"  Thermal strain  : {abs(epsilon)*100:.3f} %  <=  {epsilon_max*100:.3f} % (limit) | {'PASS' if pass_strain else 'FAIL'}")
print(f"  OVERALL         : {'PASS' if overall else 'FAIL'}")


--- VALIDATION ---
  Thermal stress  : 4.200 MPa  <=  30.0 MPa (yield)  | PASS
  Thermal strain  : 0.280 %  <=  0.800 % (limit) | PASS
  OVERALL         : PASS
